## Step 0: Load News Data and Extract Unique Source Categories

Goal: build the list of raw categories that will be normalized later.

What the next code cell does:
- loads latest-news.json from the scraping output folder
- supports both list-based and dictionary-based JSON layouts
- reads category and categories fields from each article
- handles both comma-separated strings and list values
- produces a deduplicated, sorted list of category names

Output of this step:
- unique_categories: the source category vocabulary to classify

In [39]:
import json
from pathlib import Path

# Resolve project root whether the notebook runs from repo root or word_embeding/.
project_root_candidates = [Path("."), Path("..")]
PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in project_root_candidates
        if (candidate / "scraping_system").exists() and (candidate / "word_embeding").exists()
    ),
    Path(".").resolve(),
)

news_json_path = PROJECT_ROOT / "scraping_system" / "news" / "latest-news.json"

with news_json_path.open("r", encoding="utf-8") as f:
    news_data = json.load(f)

if isinstance(news_data, list):
    news_items = news_data
elif isinstance(news_data, dict):
    for key in ("articles", "news", "items", "data"):
        if key in news_data and isinstance(news_data[key], list):
            news_items = news_data[key]
            break
    else:
        news_items = [news_data]
else:
    news_items = []

raw_categories = []
for item in news_items:
    if not isinstance(item, dict):
        continue

    for field in ("category", "categories"):
        if field not in item:
            continue

        value = item[field]
        if isinstance(value, list):
            raw_categories.extend(value)
        elif isinstance(value, str):
            raw_categories.extend(part.strip() for part in value.split(","))

unique_categories = sorted({
    c.strip() for c in raw_categories if isinstance(c, str) and c.strip()
})

print("There are {} unique categories.".format(len(unique_categories)))
print(unique_categories)

FileNotFoundError: [Errno 2] No such file or directory: '..\\scraping_system\\news\\latest-news.json'

## Step 1: Load Embedding Model (Project Root Cache)

Goal: make a word-vector model available for semantic category matching.

What the next code cell does:
- resolves the project root reliably from current working directory
- uses a shared root folder for models at models
- loads the model immediately if found
- downloads glove-wiki-gigaword-100 only when no local file exists
- saves downloaded files into the root models folder for reuse

Output of this step:
- model: ready for similarity scoring in classification steps

In [34]:
from pathlib import Path
from gensim.models import KeyedVectors
import gensim.downloader as api

MODEL_NAME = "glove-wiki-gigaword-100"

# Resolve project root from either root cwd or word_embeding cwd.
project_root_candidates = [Path("."), Path("..")]
PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in project_root_candidates
        if (candidate / "scraping_system").exists() and (candidate / "word_embeding").exists()
    ),
    Path(".").resolve(),
)

SAVE_DIR = PROJECT_ROOT / "models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PATH = SAVE_DIR / f"{MODEL_NAME}.bin"

if SAVE_PATH.exists():
    print(f"Local model found. Loading from {SAVE_PATH}")
    model = KeyedVectors.load(str(SAVE_PATH))
else:
    print(f"Local model not found. Downloading {MODEL_NAME}...")
    model = api.load(MODEL_NAME)
    print(f"Saving model to {SAVE_PATH}")
    model.save(str(SAVE_PATH))

print("Model is ready to use")

Local model not found. Downloading glove-wiki-gigaword-100...
[==================================================] 100.0% 128.1/128.1MB downloaded
Saving model to C:\Users\Mr comPuter\Downloads\news platform gp\news_platform\models\glove-wiki-gigaword-100.bin
Model is ready to use


## Step 2: Define Target Labels, Aliases, and Manual Rules

Goal: define the taxonomy used by the classifier.

What the next code cell does:
- declares canonical_labels (the final normalized categories)
- creates label_aliases (keyword sets per label)
- sets manual_overrides for known edge cases and explicit mappings

Output of this step:
- core classification rules used by both exact-match logic and embedding scoring

In [ ]:
import re
import unicodedata

canonical_labels = [
    "sport",
    "technology",
    "politics",
    "economy",
    "health",
    "culture",
    "society",
    "education",
]

label_aliases = {
    "sport": [
        "sport", "sports", "football", "basketball", "soccer", "athlete", "match",
        "competition", "team", "national team"
    ],
    "technology": [
        "technology", "tech", "digital", "innovation", "software", "ai"
    ],
    "politics": [
        "politics", "political", "government", "diplomacy", "policy", "state",
        "presidency", "national", "international", "world", "news", "intl"
    ],
    "economy": [
        "economy", "economic", "business", "finance", "industry", "market", "trade",
        "energy", "bank", "development"
    ],
    "health": [
        "health", "medical", "medicine", "hospital", "care", "wellness", "environment"
    ],
    "culture": [
        "culture", "cultural", "art", "arts", "media", "heritage"
    ],
    "society": [
        "society", "social", "community", "public", "local", "women", "woman"
    ],
    "education": [
        "education", "school", "university", "learning", "student", "encyclopedia"
    ],
}

manual_overrides = {
    "Algérie / Développement": "economy",
    "Algérie / Éducation et Technologie": "technology",
    "intl": "politics",
    "National Team": "sport",
    "الحدث": "politics",
    "فن": "culture",
    "مرأة": "society",
    "موسوعة": "education",
}

print("Step ready: labels, aliases, and optional overrides configured")

Step ready: labels, aliases, and optional overrides configured


## Step 3: Configure French/Arabic Translation and Scoring Controls

Goal: bridge multilingual source labels to the English embedding space and tune decision quality.

What the next code cell does:
- defines French to English token mappings
- defines Arabic to English token mappings
- sets Arabic text normalization pattern
- configures classification controls: threshold, min_confidence_gap, rule_boost, strong_rule_min_overlap

Output of this step:
- language mappings and scoring parameters used by helper and mapping steps

In [ ]:
french_to_english = {
    "actualite": "news",
    "nationale": "national",
    "national": "national",
    "monde": "world",
    "arabe": "arab",
    "politique": "politics",
    "gouvernement": "government",
    "presidence": "presidency",
    "societe": "society",
    "sport": "sport",
    "sports": "sports",
    "sante": "health",
    "technologie": "technology",
    "numerique": "digital",
    "education": "education",
    "ecole": "school",
    "universite": "university",
    "economie": "economy",
    "economique": "economic",
    "banque": "bank",
    "finances": "finance",
    "commerce": "trade",
    "service": "service",
    "industrie": "industry",
    "energie": "energy",
    "mines": "mines",
    "culture": "culture",
    "culturel": "cultural",
    "environnement": "environment",
    "developpement": "development",
    "international": "international",
    "internationale": "international",
    "intl": "international",
    "equipe": "team",
    "algerie": "algeria",
}

arabic_to_english = {
    "اخبار": "news",
    "خبر": "news",
    "العالم": "world",
    "عالم": "world",
    "الجزائر": "algeria",
    "جزائر": "algeria",
    "الرياضة": "sport",
    "رياضة": "sport",
    "مباريات": "match",
    "كرة": "football",
    "المنتخب": "team",
    "سياسة": "politics",
    "سياسي": "political",
    "حكومة": "government",
    "اقتصاد": "economy",
    "اقتصادي": "economic",
    "اسواق": "market",
    "تكنولوجيا": "technology",
    "تقنية": "technology",
    "ذكاء": "ai",
    "ثقافة": "culture",
    "ثقافي": "cultural",
    "فن": "art",
    "الوطني": "national",
    "وطني": "national",
    "وطنية": "national",
    "محلي": "local",
    "مجتمع": "society",
    "المرأة": "women",
    "مرأة": "women",
    "مرآة": "women",
    "تعليم": "education",
    "موسوعة": "encyclopedia",
    "صحة": "health",
    "منوعات": "variety",
    "الحدث": "event",
    "دولي": "international",
    "دولية": "international",
    "قناة": "channel",
    "النهار": "ennahar",
    "النسخة": "edition",
    "المصورة": "illustrated",
}

arabic_diacritics_pattern = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")

# Classification controls
threshold = 0.33
min_confidence_gap = 0.05
rule_boost = 0.08
strong_rule_min_overlap = 2

print("Step ready: translation dictionaries and thresholds configured")

Step ready: translation dictionaries and thresholds configured


## Step 4: Build Text Normalization and Candidate Generation Helpers

Goal: convert noisy multilingual category text into robust candidate tokens.

What the next code cell does:
- normalizes accents and Arabic writing variations
- translates French and Arabic tokens into English tokens
- splits composite labels into candidate parts and terms
- computes rule-overlap signals between category tokens and label aliases

Output of this step:
- reusable helper functions for the final mapping engine

In [ ]:
def strip_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in normalized if not unicodedata.combining(ch))


def normalize_arabic(text: str) -> str:
    return arabic_diacritics_pattern.sub("", text)


def translate_arabic_token(token: str) -> str:
    if token in arabic_to_english:
        return arabic_to_english[token]

    if token.startswith("ال") and len(token) > 2:
        without_prefix = token[2:]
        if without_prefix in arabic_to_english:
            return arabic_to_english[without_prefix]

    return token


def translate_to_english_tokens(text: str) -> str:
    lowered = text.lower()
    ascii_text = strip_accents(lowered)
    arabic_text = normalize_arabic(lowered)

    latin_tokens = re.findall(r"[a-z]+", ascii_text)
    arabic_tokens = re.findall(r"[\u0600-\u06FF]+", arabic_text)

    translated_latin = [french_to_english.get(token, token) for token in latin_tokens]
    translated_arabic = [translate_arabic_token(token) for token in arabic_tokens]

    translated = [token for token in (translated_latin + translated_arabic) if token]
    return " ".join(translated).strip()


def category_candidates(category_name: str):
    cleaned = category_name.strip().lower()
    normalized_latin = strip_accents(cleaned)
    normalized_arabic = normalize_arabic(cleaned)

    split_ready = (
        normalized_latin
        .replace("&", " and ")
        .replace("|", "/")
        .replace("-", " ")
        .replace(",", "/")
        .replace("،", "/")
        .replace("؛", "/")
    )

    parts = [p.strip() for p in split_ready.split("/") if p.strip()]
    candidates = set(parts + [normalized_latin, normalized_arabic])

    for part in parts:
        tokens = [token for token in part.split() if token]
        candidates.update(tokens)

        translated_part = translate_to_english_tokens(part)
        if translated_part:
            candidates.add(translated_part)
            candidates.update(translated_part.split())

    translated_full = translate_to_english_tokens(cleaned)
    if translated_full:
        candidates.add(translated_full)
        candidates.update(translated_full.split())

    return sorted(candidates)


def category_rule_overlap(category_name: str, aliases_by_label: dict) -> dict:
    # Compute exact token overlap signal before embedding similarity.
    translated_text = translate_to_english_tokens(category_name)
    tokens = set(category_candidates(category_name))

    if translated_text:
        tokens.update(translated_text.split())

    overlap = {}
    for label, aliases in aliases_by_label.items():
        alias_tokens = {a.lower() for a in aliases}
        overlap[label] = len(tokens.intersection(alias_tokens))

    return overlap


print("Step ready: normalization and translation helpers loaded")

Step ready: normalization and translation helpers loaded


## Step 5: Classify Categories with Hybrid Rules and Embeddings

Goal: assign each source category to one canonical label with confidence tracking.

What the next code cell does:
- filters aliases to terms available in the model vocabulary
- computes embedding similarity between source terms and label aliases
- combines semantic scores with rule-overlap boosts
- applies threshold and confidence-gap checks
- marks uncertain cases and out-of-vocabulary cases for review

Outputs of this step:
- category_mapping
- category_confidence
- out_of_vocab_categories
- low_confidence_categories

In [ ]:
import numpy as np

available_label_aliases = {
    label: [alias for alias in aliases if alias in model.key_to_index]
    for label, aliases in label_aliases.items()
}

category_mapping = {}
category_confidence = {}
low_confidence_categories = []
out_of_vocab_categories = []


def embedding_score(category_terms, label_terms):
    if not category_terms or not label_terms:
        return -1.0

    similarities = [
        model.similarity(category_term, label_term)
        for category_term in category_terms
        for label_term in label_terms
    ]

    if not similarities:
        return -1.0

    # Blend best hit with average behavior to reduce noisy single-token matches.
    return (0.6 * max(similarities)) + (0.4 * float(np.mean(similarities)))


for category in unique_categories:
    if category in manual_overrides:
        category_mapping[category] = manual_overrides[category]
        category_confidence[category] = 1.0
        continue

    candidates = category_candidates(category)
    category_terms = [term for term in candidates if term in model.key_to_index]

    # First pass: exact overlap with known aliases.
    overlap_by_label = category_rule_overlap(category, label_aliases)
    best_rule_label, best_rule_overlap = max(
        overlap_by_label.items(),
        key=lambda x: x[1],
    )

    if not category_terms:
        # Allow a strong exact-rule match when embeddings are unavailable.
        if best_rule_overlap >= strong_rule_min_overlap:
            category_mapping[category] = best_rule_label
            category_confidence[category] = 0.6
        else:
            category_mapping[category] = "other"
            category_confidence[category] = 0.0
            out_of_vocab_categories.append(category)
        continue

    label_scores = {}
    for label in canonical_labels:
        label_terms = available_label_aliases.get(label, [])
        emb_score = embedding_score(category_terms, label_terms)

        if overlap_by_label.get(label, 0) > 0 and emb_score > -1.0:
            emb_score += rule_boost * overlap_by_label[label]

        label_scores[label] = emb_score

    sorted_labels = sorted(
        label_scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    best_label, best_score = sorted_labels[0]
    second_score = sorted_labels[1][1] if len(sorted_labels) > 1 else -1.0
    confidence_gap = best_score - second_score

    if best_score >= threshold and confidence_gap >= min_confidence_gap:
        category_mapping[category] = best_label
    elif best_rule_overlap >= strong_rule_min_overlap:
        category_mapping[category] = best_rule_label
    else:
        category_mapping[category] = "other"
        low_confidence_categories.append(category)

    category_confidence[category] = round(float(best_score), 4)

print(f"Mapped {len(category_mapping)} categories")
print(f"Out-of-vocabulary categories: {len(out_of_vocab_categories)}")
print(f"Low-confidence categories: {len(low_confidence_categories)}")
print("Sample mapping with confidence:")
for source_category in list(category_mapping.keys())[:20]:
    mapped_label = category_mapping[source_category]
    score = category_confidence[source_category]
    print(f"- {source_category} -> {mapped_label} (score={score})")

if low_confidence_categories:
    print("\nLow-confidence examples:")
    for category in low_confidence_categories[:15]:
        print(f"- {category}")

Mapped 25 categories
Out-of-vocabulary categories: 0
Low-confidence categories: 3
Sample mapping with confidence:
- Basketball -> sport (score=0.9578)
- Football -> sport (score=0.9688)
- International -> politics (score=0.8897)
- L'Actualité en temps réel -> politics (score=0.7471)
- National -> politics (score=0.8929)
- National Team -> sport (score=1.0)
- Société -> society (score=0.7742)
- Uncategorized -> other (score=-0.0959)
- arab -> other (score=0.4976)
- intl -> politics (score=1.0)
- Économie -> economy (score=0.9475)
- أخبار -> politics (score=0.8638)
- أخبار الجزائر -> politics (score=0.819)
- أخبار العالم -> politics (score=0.9488)
- اقتصاد -> economy (score=0.9475)
- الجزائر -> other (score=0.3212)
- الحدث -> politics (score=1.0)
- الرياضة -> sport (score=0.9397)
- العالم -> politics (score=0.8737)
- الوطني -> politics (score=0.8929)

Low-confidence examples:
- Uncategorized
- arab
- الجزائر


## Step 6: Write Standardized Categories Back to News JSON

Goal: apply normalized labels to the dataset and persist the result.

What the next code cell does:
- extracts existing category values from each article
- maps old values through category_mapping
- writes normalized values into category and categories fields
- preserves the original top-level JSON structure
- saves updates directly to latest-news.json

Output of this step:
- latest-news.json updated in place with standardized categories

In [35]:
def extract_old_categories(article: dict):
    values = []

    for field in ("category", "categories"):
        if field not in article:
            continue

        value = article[field]
        if isinstance(value, list):
            values.extend(value)
        elif isinstance(value, str):
            values.extend(part.strip() for part in value.split(","))

    ordered_unique = []
    seen = set()

    for value in values:
        if not isinstance(value, str):
            continue

        cleaned = value.strip()
        if not cleaned or cleaned in seen:
            continue

        seen.add(cleaned)
        ordered_unique.append(cleaned)

    return ordered_unique


updated_news_items = []

for article in news_items:
    if not isinstance(article, dict):
        continue

    old_categories = extract_old_categories(article)
    if not old_categories:
        old_categories = ["uncategorized"]

    standardized_categories = sorted({
        category_mapping.get(old_category, "other")
        for old_category in old_categories
    })

    updated_article = dict(article)
    updated_article["category"] = (
        standardized_categories[0]
        if len(standardized_categories) == 1
        else standardized_categories
    )

    if "categories" in updated_article or isinstance(updated_article["category"], list):
        updated_article["categories"] = standardized_categories

    updated_news_items.append(updated_article)


def merge_updated_items(original_data, updated_items):
    if isinstance(original_data, list):
        return updated_items

    if isinstance(original_data, dict):
        updated = dict(original_data)
        for key in ("articles", "news", "items", "data"):
            if key in updated and isinstance(updated[key], list):
                updated[key] = updated_items
                return updated

    return updated_items


updated_news_data = merge_updated_items(news_data, updated_news_items)

with news_json_path.open("w", encoding="utf-8") as f:
    json.dump(updated_news_data, f, ensure_ascii=False, indent=2)

print(f"Updated JSON file in place: {news_json_path.resolve()}")
print(f"Articles updated: {len(updated_news_items)}")
print("\nOld category -> standardized category:")
for old_category in sorted(category_mapping.keys(), key=lambda x: x.lower()):
    print(f"- {old_category} -> {category_mapping[old_category]}")

NameError: name 'news_data' is not defined

## Step 7 (Optional): Review Grouped Mapping for Quality Check

Goal: quickly validate mapping quality before using the output downstream.

What the next code cell does:
- groups original categories by their final normalized label
- prints grouped values to highlight misclassifications or gaps

Output of this step:
- a human-readable audit view for manual validation

In [ ]:
grouped_mapping = {}
for old_category, new_category in category_mapping.items():
    grouped_mapping.setdefault(new_category, []).append(old_category)

print("Grouped old categories by standardized category:")
for new_category in sorted(grouped_mapping.keys()):
    old_values = sorted(grouped_mapping[new_category], key=lambda x: x.lower())
    print(f"\n{new_category}:")
    for old_category in old_values:
        print(f"  - {old_category}")

Grouped old categories by standardized category:

culture:
  - ثقافة
  - فن

economy:
  - Économie
  - اقتصاد

other:
  - arab
  - Uncategorized
  - الجزائر

politics:
  - International
  - intl
  - L'Actualité en temps réel
  - National
  - أخبار
  - أخبار الجزائر
  - أخبار العالم
  - الحدث
  - العالم
  - الوطني
  - سياسة

society:
  - Société

sport:
  - Basketball
  - Football
  - National Team
  - الرياضة
  - رياضة

technology:
  - تكنولوجيا


## Step 8: Duplicate Detection and Handling

Goal: detect duplicate and near-duplicate articles using semantic similarity, then group them while preserving sources.

What do we need this step : 
- Removes redundancy
- Prepares data for clustering and summarization
- Preserves multi source credibility


## Load Sentence Embedding Model

Goal: Use a sentence-level embedding model to represent full articles semantically.
- We need document level understanding not word level 

What will we do  : 
- loads SentenceTransformer model (all-MiniLM-L6-v2)
- caches it in the same models folder
- avoids re-downloading

Output of this step:

- sentence_model: ready to encode articles

In [ ]:
from sentence_transformers import SentenceTransformer
from pathlib import Path

# Define SAVE_DIR if not already set
if 'SAVE_DIR' not in locals():
    project_root_candidates = [Path("."), Path("..")]
    PROJECT_ROOT = next(
        (
            candidate.resolve()
            for candidate in project_root_candidates
            if (candidate / "scraping_system").exists() and (candidate / "word_embeding").exists()
        ),
        Path(".").resolve(),
    )
    SAVE_DIR = PROJECT_ROOT / "models"
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

SENTENCE_MODEL_NAME = "all-MiniLM-L6-v2"

sentence_model_path = SAVE_DIR / SENTENCE_MODEL_NAME

if sentence_model_path.exists():
    print(f"Loading sentence model from {sentence_model_path}")
    sentence_model = SentenceTransformer(str(sentence_model_path))
else:
    print(f"Downloading {SENTENCE_MODEL_NAME}...")
    sentence_model = SentenceTransformer(SENTENCE_MODEL_NAME)
    sentence_model.save(str(sentence_model_path))

print("Sentence embedding model ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence embedding model ready


## Build Article Text Representation
Goal:
Create a unified text per article for similarity comparison.

What will we do :
- combines title + content
- ensures no empty values
- produces clean text list

Output of this step:

- article_texts: list of full article texts

In [ ]:
article_texts = []

for article in updated_news_items:
    title = article.get("title", "") or ""
    content = article.get("content", "") or ""

    full_text = (title + " " + content).strip()
    article_texts.append(full_text)

print(f"Prepared {len(article_texts)} article texts")

NameError: name 'updated_news_items' is not defined

## Generate Semantic Embeddings
Goal:
Convert each article into a vector representation.

What will we do : 
- encodes all article texts
- normalizes embeddings (important for cosine similarity)

Output of this step:
- embeddings: numpy array (N × D)

In [30]:
embeddings = sentence_model.encode(
    article_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f"Generated embeddings shape: {embeddings.shape}")

Batches: 0it [00:00, ?it/s]

Generated embeddings shape: (0,)


## Group Articles by Category 
Goal:

- Reduce unnecessary comparisons and improve accuracy.

What will we do :
- groups article indices by standardized category

Output of this step:

- category_groups: dict {category → list of indices}

In [31]:
from collections import defaultdict

category_groups = defaultdict(list)

for idx, article in enumerate(updated_news_items):
    category = article.get("category", "other")
    category_groups[category].append(idx)

print("Articles grouped by category:")
for cat, indices in category_groups.items():
    print(f"- {cat}: {len(indices)} articles")

Articles grouped by category:


## Compute Similarity and Detect Duplicates
Goal:
Identify duplicate and near-duplicate articles using cosine similarity.

### Thresholds:

- 0.95 : exact duplicate

- 0.85 – 0.95 : near duplicate
What will we do :
- computes similarity within each category
- builds clusters of similar articles

Output of this step:

- duplicate_clusters: list of clusters (each cluster = list of indices)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

duplicate_clusters = []
visited = set()

for category, indices in category_groups.items():
    if not indices:
        continue

    # Extract embeddings for this category
    cat_embeddings = embeddings[indices]
    sim_matrix = cosine_similarity(cat_embeddings)

    for i in range(len(indices)):
        global_i = indices[i]

        if global_i in visited:
            continue

        cluster = [global_i]
        visited.add(global_i)

        for j in range(i + 1, len(indices)):
            global_j = indices[j]

            if global_j in visited:
                continue

            sim = sim_matrix[i][j]

            if sim >= 0.85:
                cluster.append(global_j)
                visited.add(global_j)

        duplicate_clusters.append(cluster)

print(f"Generated {len(duplicate_clusters)} clusters")

## Classify Cluster Types
Goal:
Label clusters as:

- exact_duplicate
- near_duplicate
- unique

what will we do :

- checks similarity inside cluster
- assigns cluster type

Output of this step:

- cluster_metadata

In [ ]:
cluster_metadata = []

for cluster in duplicate_clusters:
    if len(cluster) == 1:
        cluster_type = "unique"
    else:
        sims = []
        for i in range(len(cluster)):
            for j in range(i + 1, len(cluster)):
                sim = cosine_similarity(
                    [embeddings[cluster[i]]],
                    [embeddings[cluster[j]]]
                )[0][0]
                sims.append(sim)

        avg_sim = sum(sims) / len(sims) if sims else 0

        if avg_sim > 0.95:
            cluster_type = "exact_duplicate"
        else:
            cluster_type = "near_duplicate"

    cluster_metadata.append({
        "cluster": cluster,
        "type": cluster_type
    })

print("Cluster classification completed")

## Build Final Processed Articles
Goal:

Create final cleaned dataset with source preservation

What will we do : 
- selects representative article
- attaches all sources
- adds cluster metadata

Output of this step:

- processed_articles

In [ ]:
processed_articles = []

for idx, cluster_info in enumerate(cluster_metadata):
    cluster = cluster_info["cluster"]
    cluster_type = cluster_info["type"]

    representative = updated_news_items[cluster[0]]

    sources = []
    for article_idx in cluster:
        art = updated_news_items[article_idx]
        sources.append({
            "source": art.get("source"),
            "title": art.get("title"),
            "date": art.get("date")
        })

    new_article = dict(representative)
    new_article["cluster_id"] = f"cluster_{idx}"
    new_article["cluster_type"] = cluster_type
    new_article["cluster_size"] = len(cluster)
    new_article["sources"] = sources

    processed_articles.append(new_article)

print(f"Processed {len(processed_articles)} final articles")

## Save Processed Data
Goal:

Persist cleaned and deduplicated dataset

What will we do : 
- writes output to new JSON file

Output:

- deduplicated-news.json

In [ ]:
output_path = news_json_path.parent / "deduplicated-news.json"

with output_path.open("w", encoding="utf-8") as f:
    json.dump(processed_articles, f, ensure_ascii=False, indent=2)

print(f"Saved deduplicated data to {output_path.resolve()}")